In [2]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [4]:
spark = SparkSession.builder \
    .appName("Week6 Assignment") \
    .getOrCreate()

In [5]:
from google.colab import files

uploaded = files.upload()

Saving dataset.csv to dataset.csv


Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

Answer:

Driver: Creates the SparkSession, builds the DAG, schedules tasks, and collects results.

Cluster Manager: Allocates resources and manages executors.

Executor: Executes tasks, processes data, and returns results to the Driver.

Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

Answer:

Spark does not execute transformations immediately. It records them and creates a DAG. Execution starts only when an action (show(), count(), etc.) is called. This optimization reduces unnecessary computations and improves performance.

Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [6]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/content/dataset.csv")

In [7]:
df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- tax: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- final_price: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- vendor: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- returned: string (nullable = tr

In [8]:
df.show(5)

+----------+---------+--------+-------+----------+--------+----------+-------+------+--------+--------+----------+----------+--------+--------+-------+-------------+-----------+--------------+---------+-------+---------+------------+------+--------+
|product_id| old_name|category|  price|base_price|  amount|    status|user_id|region|priority|order_id|order_date| ship_date|quantity|discount|    tax|shipping_cost|final_price|payment_method|     city|country|   vendor|warehouse_id|rating|returned|
+----------+---------+--------+-------+----------+--------+----------+-------+------+--------+--------+----------+----------+--------+--------+-------+-------------+-----------+--------------+---------+-------+---------+------------+------+--------+
|   P000001|Product_1|  Beauty| 345.88|    356.58| 2421.16| Completed| U83704|  East|  Medium| O100001|2022-04-17|2022-11-24|       7|    0.03| 435.81|       169.95|    3026.92|   Credit Card|Hyderabad|  India|Vendor_36|         W11|     2|     Yes|


Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

| Feature | CSV | Parquet |
|---------|-----|----------|
| Storage Format | Row-based | Column-based |
| Schema | Does not store schema | Stores schema |
| File Size | Larger | Smaller (Compressed) |
| Read Performance | Slower | Faster |
| Compression | No | Yes |
| Predicate Pushdown | Not Supported | Supported |

**Performance Insight:**

- CSV stores data row by row, so Spark reads the entire file even if only a few columns are needed.
- Parquet stores data column by column, allowing Spark to read only the required columns.
- Parquet also supports **Predicate Pushdown**, which loads only the filtered rows into memory.
- Therefore, Parquet provides better performance, reduced disk I/O, and lower memory usage, making it ideal for large datasets.

Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.


In [9]:
df.filter(col("category") == "Electronics") \
  .select("product_id", "price") \
  .show()

+----------+-------+
|product_id|  price|
+----------+-------+
|   P000006|3707.43|
|   P000016| 152.13|
|   P000020|4642.98|
|   P000024| 361.16|
|   P000025| 2973.4|
|   P000031|2317.97|
|   P000040|1021.31|
|   P000043|2492.58|
|   P000045|1361.14|
|   P000051|1086.71|
|   P000056|2915.96|
|   P000067|3773.03|
|   P000073|2343.06|
|   P000074|2522.34|
|   P000077|4219.58|
|   P000096|2278.96|
|   P000106|3379.54|
|   P000109|1163.56|
|   P000132|3106.58|
|   P000136|1031.72|
+----------+-------+
only showing top 20 rows


Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In [10]:
df = df.withColumnRenamed("old_name", "new_name") \
       .withColumn("price", col("price").cast("double"))

df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- tax: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- final_price: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- vendor: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- returned: string (nullable = tr

Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Answer:

Spark maintains a Lineage Graph (DAG). If an executor fails, Spark recomputes only the lost partitions using the DAG instead of reprocessing the entire dataset.

Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [11]:
df.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
).show()

+----------+-----------+-----------+-------+----------+--------+---------+-------+------+--------+--------+----------+----------+--------+--------+-------+-------------+-----------+--------------+---------+-------+---------+------------+------+--------+
|product_id|   new_name|   category|  price|base_price|  amount|   status|user_id|region|priority|order_id|order_date| ship_date|quantity|discount|    tax|shipping_cost|final_price|payment_method|     city|country|   vendor|warehouse_id|rating|returned|
+----------+-----------+-----------+-------+----------+--------+---------+-------+------+--------+--------+----------+----------+--------+--------+-------+-------------+-----------+--------------+---------+-------+---------+------------+------+--------+
|   P000001|  Product_1|     Beauty| 345.88|    356.58| 2421.16|Completed| U83704|  East|  Medium| O100001|2022-04-17|2022-11-24|       7|    0.03| 435.81|       169.95|    3026.92|   Credit Card|Hyderabad|  India|Vendor_36|         W11| 

Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Answer:

Predicate Pushdown applies filter conditions while reading Parquet files. Only matching rows are loaded into memory, reducing disk I/O and improving performance.

Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [12]:
df = df.withColumn(
    "calculated_final_price",
    col("base_price") * 1.18
)

df.select("base_price", "calculated_final_price").show(5)

+----------+----------------------+
|base_price|calculated_final_price|
+----------+----------------------+
|    356.58|    420.76439999999997|
|    108.19|              127.6642|
|   1400.08|    1652.0943999999997|
|   4251.97|             5017.3246|
|   4174.86|             4926.3348|
+----------+----------------------+
only showing top 5 rows


Q11: What is the difference between Transformations and Actions? Provide two examples of each.

Answer
Transformations

filter(),
select()

Actions

show(),
count()

Transformations are lazily evaluated, while actions trigger execution.

Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [16]:
from pyspark.sql.functions import col

# Save DataFrame as Parquet
df.write.mode("overwrite").parquet("week6_parquet")

# Read Parquet file
df_parquet = spark.read.parquet("week6_parquet")

# Filter null values
filtered_df = df_parquet.filter(col("user_id").isNotNull())

# Show result
filtered_df.show(5)

# Save filtered data as CSV
filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("week6_output_csv")

print("Parquet file created successfully.")
print("Filtered CSV file saved successfully.")

+----------+---------+-----------+-------+----------+--------+----------+-------+------+--------+--------+----------+----------+--------+--------+-------+-------------+-----------+--------------+---------+-------+---------+------------+------+--------+----------------------+
|product_id| new_name|   category|  price|base_price|  amount|    status|user_id|region|priority|order_id|order_date| ship_date|quantity|discount|    tax|shipping_cost|final_price|payment_method|     city|country|   vendor|warehouse_id|rating|returned|calculated_final_price|
+----------+---------+-----------+-------+----------+--------+----------+-------+------+--------+--------+----------+----------+--------+--------+-------+-------------+-----------+--------------+---------+-------+---------+------------+------+--------+----------------------+
|   P000001|Product_1|     Beauty| 345.88|    356.58| 2421.16| Completed| U83704|  East|  Medium| O100001|2022-04-17|2022-11-24|       7|    0.03| 435.81|       169.95|    

Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

Client Mode

*   Driver runs on the client machine.
*   Client must stay connected.

Cluster Mode


*   Driver runs inside the cluster.
*  Application continues even if the client disconnects.


Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [15]:
df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
).show()

+----------+----------+-----------+-------+----------+--------+----------+-------+------+--------+--------+----------+----------+--------+--------+-------+-------------+-----------+--------------+---------+-------+---------+------------+------+--------+----------------------+
|product_id|  new_name|   category|  price|base_price|  amount|    status|user_id|region|priority|order_id|order_date| ship_date|quantity|discount|    tax|shipping_cost|final_price|payment_method|     city|country|   vendor|warehouse_id|rating|returned|calculated_final_price|
+----------+----------+-----------+-------+----------+--------+----------+-------+------+--------+--------+----------+----------+--------+--------+-------+-------------+-----------+--------------+---------+-------+---------+------------+------+--------+----------------------+
|   P000002| Product_2|      Books|  91.96|    108.19|  183.92| Completed|   NULL|  West|    High| O100002|2022-03-31|2022-07-10|       2|    0.15|  33.11|       445.38|

Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

Answer:

.show(5) displays only the first five rows, making it safe for exploring large datasets.

.collect() retrieves the entire dataset to the Driver's memory, which can cause out-of-memory errors when working with large datasets.